pip install torch
pip install opencv-python

In [19]:
import torch
from torch import nn,optim,accelerator
from torch.utils.data import Dataset,DataLoader
import numpy as np
import cv2
import os
from random import random

if accelerator.is_available():
    device = accelerator.current_aclerator()
else:
    device="cpu"

print(device)

cpu


In [20]:
# 전체 데이터 30%정도는 학습용X, 테스트용
def getImg(folder,w,h,yData):
    trainData =[]
    trainLabel =[]
    testData =[]
    testLabel =[]

    for i, f in enumerate(os.listdir(folder)):
        data = cv2.imread(folder+f,cv2.IMREAD_COLOR)
        data = cv2.resize(data,(w,h))
        if random() < 0.3:
            testData.append(np.array(data,dtype=np.float32))
            testLabel.append(yData[i])
        else:
            trainData.append(np.array(data,dtype=np.float32))
            trainLabel.append(yData[i]) 
 
        
    return trainData,trainLabel,testData,testLabel

In [21]:
class LeeBunsikDataset(Dataset):
    def __init__(self,data,label, numClasses):
        super().__init__()
        self.datas = torch.from_numpy(np.array(data)) # 18 x 50 행 x 100열 x 3색
        self.datas = self.datas.permute(0,3,1,2) # 18 x 50 x 100 x 3 -> 18 x 3 x 50 x 100
        self.labels = torch.from_numpy(np.array(label))
        self.labels = torch.nn.functional.one_hot(self.labels, numClasses)
        self.labels= self.labels.type(torch.float32)

        print(self.labels[0])

        # print(len(self.datas)) #전체 데이터 수
        # print(self.datas[0]) # 0번째 데이터

        # print(len(self.datas[0])) # 0번째 데이터 행 수
        # print(self.datas[0][0]) # 0번째 데이터의 0번째 행

        
        # print(len(self.datas[0][0])) # 0번째 데이터 0번째 행 열수
        # print(self.datas[0][0][0]) # 0번째 데이터의 0번째 행 0번 열

        # print(len(self.datas[0][0][0])) # 0번째 데이터 0번째 행 0번 열의 R,G,B로 구성
        # print(self.datas[0][0][0][0]) # 0번째 데이터의 0번째 행 0번 열의 파란색값

    def __len__(self):
        return len(self.datas) #전체 갯수 리턴

    def __getitem__(self,index):
        return self.datas[index], self.labels[index] # (데이터,라벨) 튜플형태로 리턴

In [22]:
label =["떡볶이","오뎅","김밥","튀김","순대"]
yData= [0,0,0,0,0, 1,1,1,1,1, 2,2,2,2,2, 3,3,3,3,3, 4,4,4,4,4]
trainData, trainLabel, testData, testLabel = getImg("C:/PythoneWorkspace/ANN/student02_08/Feb02_2_DeepLearning/bunsikMenu/",100,50,yData)

trainDataset = LeeBunsikDataset(trainData,trainLabel,5)
testDataset = LeeBunsikDataset(testData,testLabel,5)
trainDataLoader=DataLoader(trainDataset,3,True)  # trainDataset에서 3개씩(batch size)  순서 섞어서
testDataLoader=DataLoader(testDataset,3,True)




tensor([1., 0., 0., 0., 0.])
tensor([0., 1., 0., 0., 0.])


In [23]:
class LeeBunsikCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.kbcnn = nn.Sequential(
            nn.Conv2d(3,1000,5),  # in 채널 (R,G,B), out채널, 5
            nn.ReLU(),
            nn.Conv2d(1000,500,5), 
            nn.ReLU(),
            nn.Conv2d(500,100,5), 
            nn.ReLU(),
            nn.Conv2d(100,50,5), 
            nn.ReLU(),
        )

        self.f = nn.Flatten()
        
        
        self.nn = nn.Sequential(
            nn.Linear(142800, 10), # 19200개가 한 줄로 들어옵니다.
            nn.ReLU(),
            nn.Linear(10, 5)     # 최종 메뉴 5개
        )

    def forward(self, x):
        # x의 데이터 타입을 실수형(float)으로 변환하여 전달
        myModel = self.kbcnn(x) 
        myModel = self.f(myModel)
        myModel = self.nn(myModel)
        return myModel

In [24]:
model = LeeBunsikCNN().to(device)

lossFn = nn.CrossEntropyLoss()
o = optim.Adam(model.parameters(),lr=0.001)

In [25]:
model.train()
for epoch in range(10):
    for x,y  in trainDataLoader:
        x = x.to(device)
        y = y.to(device)

        predY = model(x)
        l = lossFn(predY,y)

        o.zero_grad()
        l.backward()
        o.step()

print(l.item())

1.7864227294921875


In [26]:
# 정확도 테스트
# 전체 데이터 25개
#   70%정도는 학습용으로 사용
#   30%정도는 테스트용으로 빼둠
# 그 테스트용 데이터가 떡볶이인데, AI가 떡볶이라고 예측

model.eval()
size = len(testDataLoader.dataset) # 테스트용 데이터 전체 개수
# print(size)
ok = 0 # 제대로 예측한 개수
with torch.no_grad():  #GD할 필요 없으니
 for x,y in testDataLoader:
  x = x.to(device)
  y = y.to(device)

  predY = model(x)
#   print(predY)
#   print(y)
#   print(predY.argmax(1) == y.argmax(1))
#   print((predY.argmax(1) == y.argmax(1)).type(torch.float32).sum().item())
  ok += (predY.argmax(1) == y.argmax(1)).type(torch.float32).sum().item()
  print(ok/size)
  print("-----")

0.0
-----
0.0
-----


In [27]:
f = cv2.imread("./test.png",cv2.IMREAD_COLOR)
f = cv2.resize(f,(100,50))
f = np.array([f],dtype=np.float32)
predData = torch.from_numpy(f).permute(0,3,1,2)
result = model(predData)
result = nn.Softmax(dim=1)(result)
result = result.argmax().item()
print(label[result])

순대
